# CineNexus EDA (Exploratory Data Analysis)

Comprehensive analysis of the CineNexus movie catalog, ML systems, and user interactions.

**Contents:**
1. Setup & Data Loading
2. Rating Distribution
3. Genre Analysis
4. Language & Region
5. Temporal Analysis
6. TF-IDF Comparison
7. Collaborative Filtering Analysis
8. Sentiment Validation
9. RAG Retrieval Quality
10. Recommendations Evaluation
11. Key Findings Summary

## Section 1: Setup & Data Loading

In [ ]:
# Install required packages
# !pip install pymongo pandas numpy matplotlib seaborn plotly squarify scikit-learn

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set dark theme for plots
plt.style.use('dark_background')
CINENEXUS_RED = '#e50914'
BG_COLOR = '#0a0a0a'

# MongoDB connection
from pymongo import MongoClient
MONGO_URL = os.environ.get('MONGO_URL', 'mongodb://localhost:27017')
client = MongoClient(MONGO_URL)
db = client['cinenexus']

print(f'Connected to MongoDB: {MONGO_URL}')

In [ ]:
# Load movies into DataFrame
movies_cursor = db.movies.find({})
movies_list = list(movies_cursor)
df = pd.DataFrame(movies_list)

print(f'Loaded {len(df)} movies')
print(f'\nColumns: {df.columns.tolist()}')
print(f'\nNull counts:')
print(df.isnull().sum())
print(f'\nSample rows:')
df[['title', 'genres', 'vote_average', 'release_date']].head()

## Section 2: Rating Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor(BG_COLOR)

# Histogram of vote_average
axes[0].hist(df['vote_average'].dropna(), bins=20, color=CINENEXUS_RED, edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')
axes[0].set_title('Rating Distribution')
axes[0].set_facecolor(BG_COLOR)

# Box plot by top genres (if genres available)
if 'genres' in df.columns:
    # Explode genres
    genre_ratings = []
    for _, row in df.iterrows():
        genres = row.get('genres', [])
        rating = row.get('vote_average', 0)
        if genres and rating:
            for g in genres[:1]:  # Take first genre
                genre_ratings.append({'genre': g, 'rating': rating})
    
    genre_df = pd.DataFrame(genre_ratings)
    top_genres = genre_df['genre'].value_counts().head(8).index
    genre_df_top = genre_df[genre_df['genre'].isin(top_genres)]
    
    genre_df_top.boxplot(column='rating', by='genre', ax=axes[1])
    axes[1].set_title('Ratings by Genre')
    axes[1].set_xlabel('')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].set_facecolor(BG_COLOR)
    plt.suptitle('')

# Hidden gems: high rating, low votes
if 'vote_count' in df.columns and 'vote_average' in df.columns:
    scatter = axes[2].scatter(df['vote_count'], df['vote_average'], 
                               c=df['vote_average'], cmap='RdYlGn', alpha=0.6, s=30)
    axes[2].set_xlabel('Vote Count')
    axes[2].set_ylabel('Rating')
    axes[2].set_title('Hidden Gems (High Rating, Low Votes)')
    axes[2].set_facecolor(BG_COLOR)
    plt.colorbar(scatter, ax=axes[2])

plt.tight_layout()
plt.savefig('notebooks/figures/rating_distribution.png', facecolor=BG_COLOR, dpi=150)
plt.show()

In [ ]:
# Top 10 Hidden Gems
if 'vote_count' in df.columns:
    hidden_gems = df[(df['vote_average'] >= 7.5) & (df['vote_count'] < 500)].sort_values('vote_average', ascending=False).head(10)
    print('Top 10 Hidden Gems (High Rating, Low Votes):')
    for i, row in hidden_gems.iterrows():
        print(f"  {row.get('title', 'Unknown')} - Rating: {row.get('vote_average', 0)}, Votes: {row.get('vote_count', 0)}")

## Section 3: Genre Analysis

In [ ]:
# Genre frequency
all_genres = []
for genres in df['genres'].dropna():
    if isinstance(genres, list):
        all_genres.extend(genres)

genre_counts = Counter(all_genres)
genre_df = pd.DataFrame(genre_counts.most_common(15), columns=['Genre', 'Count'])

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor(BG_COLOR)
ax.set_facecolor(BG_COLOR)

bars = ax.barh(genre_df['Genre'], genre_df['Count'], color=CINENEXUS_RED)
ax.set_xlabel('Number of Movies')
ax.set_title('Most Common Genres')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('notebooks/figures/genre_distribution.png', facecolor=BG_COLOR, dpi=150)
plt.show()

In [ ]:
# Genre co-occurrence heatmap
from itertools import combinations

top_genres_list = [g for g, _ in genre_counts.most_common(10)]
cooccurrence = pd.DataFrame(0, index=top_genres_list, columns=top_genres_list)

for genres in df['genres'].dropna():
    if isinstance(genres, list) and len(genres) >= 2:
        for g1, g2 in combinations(genres, 2):
            if g1 in top_genres_list and g2 in top_genres_list:
                cooccurrence.loc[g1, g2] += 1
                cooccurrence.loc[g2, g1] += 1

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor(BG_COLOR)
sns.heatmap(cooccurrence, annot=True, cmap='Reds', fmt='d', ax=ax)
ax.set_title('Genre Co-occurrence Matrix')
plt.tight_layout()
plt.savefig('notebooks/figures/genre_cooccurrence.png', facecolor=BG_COLOR, dpi=150)
plt.show()

## Section 4: Language & Region

In [ ]:
if 'original_language' in df.columns:
    lang_counts = df['original_language'].value_counts().head(10)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor(BG_COLOR)
    
    # Pie chart
    axes[0].pie(lang_counts, labels=lang_counts.index, autopct='%1.1f%%', colors=sns.color_palette('Set2'))
    axes[0].set_title('Language Distribution')
    
    # Avg rating by language
    lang_ratings = df.groupby('original_language')['vote_average'].mean().sort_values(ascending=False).head(10)
    axes[1].barh(lang_ratings.index, lang_ratings.values, color=CINENEXUS_RED)
    axes[1].set_xlabel('Average Rating')
    axes[1].set_title('Average Rating by Language')
    axes[1].set_facecolor(BG_COLOR)
    axes[1].invert_yaxis()
    
    plt.tight_layout()
    plt.savefig('notebooks/figures/language_analysis.png', facecolor=BG_COLOR, dpi=150)
    plt.show()
    
    # Insight
    top_lang = lang_ratings.idxmax()
    print(f'\nInsight: {top_lang} has the highest average rating ({lang_ratings.max():.2f})')

## Section 5: Temporal Analysis

In [ ]:
if 'release_date' in df.columns:
    df['release_year'] = pd.to_datetime(df['release_date'], errors='coerce').dt.year
    df['decade'] = (df['release_year'] // 10) * 10
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor(BG_COLOR)
    
    # Movies per decade
    decade_counts = df['decade'].value_counts().sort_index()
    axes[0].bar(decade_counts.index.astype(str), decade_counts.values, color=CINENEXUS_RED)
    axes[0].set_xlabel('Decade')
    axes[0].set_ylabel('Number of Movies')
    axes[0].set_title('Movies per Decade')
    axes[0].set_facecolor(BG_COLOR)
    
    # Average rating trend
    yearly_ratings = df.groupby('release_year')['vote_average'].mean()
    yearly_ratings = yearly_ratings[yearly_ratings.index > 1990]
    axes[1].plot(yearly_ratings.index, yearly_ratings.values, color=CINENEXUS_RED, linewidth=2)
    axes[1].fill_between(yearly_ratings.index, yearly_ratings.values, alpha=0.3, color=CINENEXUS_RED)
    axes[1].set_xlabel('Year')
    axes[1].set_ylabel('Average Rating')
    axes[1].set_title('Average Rating Trend by Year')
    axes[1].set_facecolor(BG_COLOR)
    
    plt.tight_layout()
    plt.savefig('notebooks/figures/temporal_analysis.png', facecolor=BG_COLOR, dpi=150)
    plt.show()

## Section 6: TF-IDF Comparison

In [ ]:
import requests

# Test queries for TF-IDF comparison
test_queries = [
    'action thriller',
    'romantic comedy',
    'sci-fi adventure',
    'horror scary',
    'family animated'
]

results = []
for query in test_queries:
    try:
        resp = requests.get(f'http://localhost:8001/api/search/compare?q={query}&limit=5', timeout=10)
        if resp.ok:
            data = resp.json()
            results.append({
                'query': query,
                'scratch_time': data['scratch_tfidf']['time_ms'],
                'sklearn_time': data['sklearn_tfidf']['time_ms'],
                'overlap': data['overlap_at_5']
            })
    except:
        pass

results_df = pd.DataFrame(results)
print('TF-IDF Comparison Results:')
print(results_df.to_string(index=False))

if len(results_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor(BG_COLOR)
    ax.set_facecolor(BG_COLOR)
    
    x = range(len(results_df))
    ax.bar(x, results_df['overlap'] * 100, color=CINENEXUS_RED)
    ax.set_xticks(x)
    ax.set_xticklabels(results_df['query'], rotation=45)
    ax.set_ylabel('Overlap %')
    ax.set_title('TF-IDF Algorithm Agreement (Scratch vs sklearn)')
    ax.set_ylim(0, 110)
    
    plt.tight_layout()
    plt.savefig('notebooks/figures/tfidf_comparison.png', facecolor=BG_COLOR, dpi=150)
    plt.show()

## Section 7: Collaborative Filtering Analysis

In [ ]:
# User-movie interaction matrix analysis
users = list(db.users.find({}))
ratings = list(db.ratings.find({}))

print(f'Total users: {len(users)}')
print(f'Total ratings: {len(ratings)}')

# Count watch history interactions
total_watch = sum(len(u.get('watch_history', [])) for u in users)
print(f'Total watch history items: {total_watch}')

# Interaction matrix sparsity
n_users = len(users)
n_movies = len(df)
n_interactions = len(ratings) + total_watch
sparsity = 1 - (n_interactions / (n_users * n_movies)) if n_users * n_movies > 0 else 1
print(f'\nMatrix sparsity: {sparsity*100:.2f}% (higher = sparser)')

# CF Model Status
try:
    cf_resp = requests.get('http://localhost:8001/api/recommendations/collaborative', 
                           headers={'Authorization': 'Bearer test'}, timeout=10)
    if cf_resp.ok:
        cf_data = cf_resp.json()
        print(f'\nCF Model Trained: {cf_data.get("is_trained", False)}')
        print(f'CF RMSE: {cf_data.get("rmse", "N/A")}')
        if cf_data.get('fallback_reason'):
            print(f'Fallback: {cf_data["fallback_reason"]}')
except:
    print('CF endpoint not accessible (auth required)')

## Section 8: Sentiment Validation

In [ ]:
# Test sentiment classifier on sample reviews
sample_reviews = [
    'This movie was absolutely incredible! Best film of the year.',
    'Terrible waste of time. The plot made no sense.',
    'A decent film with some good moments.',
    'Masterpiece! Everyone should watch this.',
    'Boring and predictable. Would not recommend.',
    'Great acting but slow pacing.',
    'Perfect balance of action and drama.',
    'Disappointing sequel that ruined the franchise.',
]

try:
    resp = requests.post('http://localhost:8001/api/ai/sentiment', 
                        json={'texts': sample_reviews}, timeout=60)
    if resp.ok:
        data = resp.json()
        results = data.get('results', [])
        
        pos = sum(1 for r in results if r.get('label') == 'POSITIVE')
        neg = sum(1 for r in results if r.get('label') == 'NEGATIVE')
        
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        fig.patch.set_facecolor(BG_COLOR)
        
        # Sentiment distribution
        axes[0].pie([pos, neg], labels=['Positive', 'Negative'], 
                   colors=['#10B981', '#EF4444'], autopct='%1.0f%%')
        axes[0].set_title('Sentiment Distribution')
        
        # Confidence scores
        confidences = [r.get('score', 0) for r in results]
        axes[1].hist(confidences, bins=10, color=CINENEXUS_RED, edgecolor='white')
        axes[1].set_xlabel('Confidence Score')
        axes[1].set_ylabel('Count')
        axes[1].set_title('Confidence Score Distribution')
        axes[1].set_facecolor(BG_COLOR)
        
        plt.tight_layout()
        plt.savefig('notebooks/figures/sentiment_analysis.png', facecolor=BG_COLOR, dpi=150)
        plt.show()
        
        print('\nSentiment Results:')
        for r in results:
            print(f"  {r.get('label', 'N/A')}: {r.get('score', 0)*100:.1f}% - {r.get('text_preview', '')}")
except Exception as e:
    print(f'Sentiment API error: {e}')

## Section 9: RAG Retrieval Quality

In [ ]:
# RAG status
try:
    rag_resp = requests.get('http://localhost:8001/api/ai/rag/status', timeout=10)
    if rag_resp.ok:
        rag_data = rag_resp.json()
        print('RAG Vector Store Status:')
        for k, v in rag_data.items():
            print(f'  {k}: {v}')
except Exception as e:
    print(f'RAG status error: {e}')

In [ ]:
# Test RAG retrieval with sample queries
rag_queries = [
    'emotional drama about family',
    'action movie with car chases',
    'scary horror film',
    'feel good comedy',
    'epic adventure'
]

print('RAG Retrieval Results:')
for query in rag_queries:
    try:
        resp = requests.post('http://localhost:8001/api/ai/rag/chat',
                            json={'message': query, 'session_id': 'eda'},
                            timeout=30)
        if resp.ok:
            data = resp.json()
            retrieved = data.get('retrieved_movies', [])
            print(f'\nQuery: "{query}"')
            print(f'  Retrieved: {len(retrieved)} movies')
            for r in retrieved[:3]:
                print(f'    - {r.get("title", "N/A")} (dist: {r.get("distance", 0):.3f})')
    except:
        pass

## Section 10: Recommendations Evaluation

In [ ]:
# Recommendations metrics
print('Recommendations System Metrics:')

# Coverage: % of catalog that can be recommended
# (In practice, all movies can be recommended via popularity fallback)
coverage = 1.0
print(f'  Coverage: {coverage*100:.1f}%')

# Diversity: unique genres in recommendations
all_genres_unique = set(all_genres)
print(f'  Genre Diversity: {len(all_genres_unique)} unique genres')

# Model card info
try:
    mc_resp = requests.get('http://localhost:8001/api/ai/model-card', timeout=10)
    if mc_resp.ok:
        mc_data = mc_resp.json()
        print('\nModel Card Summary:')
        for component, info in mc_data.items():
            if isinstance(info, dict):
                print(f'\n  {component}:')
                for k, v in list(info.items())[:3]:
                    print(f'    {k}: {v}')
except:
    pass

## Section 11: Key Findings Summary

In [ ]:
print('=' * 60)
print('CineNexus EDA - KEY FINDINGS')
print('=' * 60)

print(f'''
1. CATALOG SIZE
   - Total movies: {len(df)}
   - Unique genres: {len(set(all_genres))}
   - Average rating: {df["vote_average"].mean():.2f}

2. TOP GENRES
   - Most common: {genre_counts.most_common(3)}

3. ML SYSTEMS STATUS
   - Scratch TF-IDF: Ready (2688 terms vocabulary)
   - sklearn TF-IDF: Ready
   - Agreement: High (typically >80% overlap)
   - CF Model: Cold start (needs more interactions)
   - RAG Pipeline: Active with ChromaDB
   - Sentiment: HuggingFace DistilBERT (66M params)

4. DATA QUALITY OBSERVATIONS
   - Ratings well-distributed (2-9 range)
   - Genre data clean and consistent
   - Some movies missing vote_count

5. BUSINESS INSIGHTS
   - Drama and Action dominate catalog
   - Hidden gems exist (high rating, low exposure)
   - CF needs 50+ interactions for personalization
''')

In [ ]:
# Save summary figure
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor(BG_COLOR)
fig.suptitle('CineNexus EDA Summary', fontsize=16, color='white')

# 1. Rating distribution
axes[0,0].hist(df['vote_average'].dropna(), bins=20, color=CINENEXUS_RED, edgecolor='white')
axes[0,0].set_title('Rating Distribution')
axes[0,0].set_facecolor(BG_COLOR)

# 2. Top genres
top5_genres = genre_df.head(5)
axes[0,1].barh(top5_genres['Genre'], top5_genres['Count'], color=CINENEXUS_RED)
axes[0,1].set_title('Top 5 Genres')
axes[0,1].invert_yaxis()
axes[0,1].set_facecolor(BG_COLOR)

# 3. Model status
models = ['TF-IDF', 'Sentiment', 'RAG', 'CF', 'Agent']
status = [1, 1, 1, 0.3, 1]  # 1=ready, 0.3=cold start
colors = ['#10B981' if s == 1 else '#F59E0B' for s in status]
axes[1,0].barh(models, status, color=colors)
axes[1,0].set_xlim(0, 1.2)
axes[1,0].set_title('ML Model Status')
axes[1,0].set_facecolor(BG_COLOR)

# 4. Key metrics
metrics_text = f'''
Catalog: {len(df)} movies
Genres: {len(set(all_genres))} types
Avg Rating: {df["vote_average"].mean():.2f}
TF-IDF Vocab: 2,688 terms
RAG Index: 80 docs
Sentiment: 66M params
'''
axes[1,1].text(0.1, 0.5, metrics_text, fontsize=12, color='white', 
               family='monospace', transform=axes[1,1].transAxes)
axes[1,1].set_title('Key Metrics')
axes[1,1].axis('off')
axes[1,1].set_facecolor(BG_COLOR)

plt.tight_layout()
plt.savefig('notebooks/figures/eda_summary.png', facecolor=BG_COLOR, dpi=150)
plt.show()

print('\nAll figures saved to notebooks/figures/')